In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
pip install google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.0/306.0 kB 10.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [4]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [5]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


In [6]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [7]:
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


In [8]:
runner=InMemoryRunner(agent=root_agent)
print("runner is running")

runner is running


In [9]:
response = await runner.run_debug(
    "What is Agent Development Kit from Google? What languages is the SDK available in?"
)


 ### Created new session: debug_session_id

User > What is Agent Development Kit from Google? What languages is the SDK available in?
helpful_assistant > The Agent Development Kit (ADK) from Google is an open-source framework designed to simplify the development, debugging, and deployment of AI agents and multi-agent systems. It aims to make building these complex systems feel more like traditional software development, offering precise control over agent behavior and orchestration. ADK is built to be flexible, allowing developers to use various models and deploy agents in different environments, though it is optimized for Google Cloud and its Gemini models.

The ADK SDK is available in the following programming languages:
*   Python
*   TypeScript
*   Go
*   Java


In [10]:
response = await runner.run_debug("how this adk push on github")


 ### Continue session: debug_session_id

User > how this adk push on github
helpful_assistant > The Agent Development Kit (ADK) is "pushed" to GitHub through its open-source model, meaning contributions and code are managed via GitHub repositories. If you want to contribute to the ADK, you would typically follow these steps:

1.  **Sign the Contributor License Agreement (CLA):** Before you can contribute, you need to sign a CLA, which grants Google permission to use and redistribute your contributions. This can be done at [https://cla.developers.google.com/](https://cla.developers.google.com/).
2.  **Fork the Repository:** You'll need to fork the relevant ADK repository (e.g., `google/adk-python`, `google/adk-js`, `google/adk-go`, `google/adk-java`, or `google/adk-docs`) to your own GitHub account.
3.  **Create a New Branch:** Make your changes on a new branch within your forked repository. This keeps your changes separate from the main development branch.
4.  **Make Your Changes:** I

In [11]:
response = await runner.run_debug("what is cost of sheep cruise")


 ### Continue session: debug_session_id

User > what is cost of sheep cruise
helpful_assistant > The cost of a "sheep cruise" generally refers to a **Bighorn Sheep Cruise** on Canyon Lake, Arizona, offered by the Dolly Steamboat. The pricing for this specific cruise is as follows:

*   **Adults (13 and over):** $55
*   **Children (5-12):** $35
*   **Small Children (1-4):** $25

These prices do not include tax and gratuity. This special event includes a classroom session about bighorn sheep prior to the cruise, and the cruise itself takes place on Canyon Lake where participants can view wild bighorn sheep.

It's important to note that general cruise costs can vary significantly. For larger ships, a 5-night cruise might start around $2500 per person, not including taxes, fees, port charges, shore excursions, onboard expenses, and gratuities. Smaller ship or river cruises can range from a few thousand dollars up to $10,000 per person.

For the Bighorn Sheep Cruise specifically, reservati